# Credit Card Fraud Detection — End-to-End Analysis

This notebook demonstrates a complete fraud detection pipeline on imbalanced data,
covering the key topics an AI engineer should be able to discuss:

1. **Understanding the imbalance** — why standard accuracy is meaningless
2. **Precision vs Recall tradeoff** — the business cost of false negatives vs false positives
3. **Threshold tuning** — why 0.5 is almost never the right threshold
4. **ROC vs PR curves** — why PR curves are more informative for imbalanced data

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

from src.preprocess import load_data, explore_data, preprocess
from src.train import get_models, cross_validate_models, train_final_model
from src.evaluate import (
    plot_precision_recall_tradeoff, tune_threshold,
    plot_roc_vs_pr_curve, full_evaluation
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load & Explore the Data

In [ ]:
df = load_data('../creditcard.csv')
stats = explore_data(df)

In [ ]:
# Visualize the extreme class imbalance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
counts = df['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution (Linear Scale)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=11)

# Log scale to see both
axes[1].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
axes[1].set_yscale('log')
axes[1].set_title('Class Distribution (Log Scale)')
axes[1].set_ylabel('Count (log)')

# Transaction amount by class
df.boxplot(column='Amount', by='Class', ax=axes[2])
axes[2].set_title('Transaction Amount by Class')
axes[2].set_xticklabels(['Legitimate', 'Fraud'])
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"\nImbalance ratio: 1 fraud per {counts[0]//counts[1]} legitimate transactions")
print(f"A naive 'predict all legitimate' classifier would get {counts[0]/len(df):.4%} accuracy!")

### Why accuracy is useless here

With only ~0.17% fraud, a model that **always predicts legitimate** gets 99.83% accuracy.
That's why we need metrics that focus on the minority class: precision, recall, F1, and AUPRC.

## 2. Preprocess & Handle Imbalance

In [ ]:
# Preprocess with SMOTE oversampling
result = preprocess(df, test_size=0.2, apply_smote=True, smote_strategy=0.5)

X_train, X_test = result['X_train'], result['X_test']
y_train, y_test = result['y_train'], result['y_test']

# Convert to numpy if needed
if hasattr(X_train, 'values'):
    X_train_np = X_train.values
else:
    X_train_np = X_train
X_test_np = X_test.values
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_test_np = y_test.values

### SMOTE (Synthetic Minority Over-sampling Technique)

SMOTE creates synthetic fraud samples by interpolating between existing fraud examples in feature space.
We only apply it to the **training set** — the test set stays untouched to reflect real-world distribution.

**`sampling_strategy` parameter:** A value of 0.5 means the minority class will be resampled to 50% *of the majority class size* (i.e. a 1:2 ratio), **not** a 50/50 split. Use `1.0` for fully balanced classes.

**Don't double-correct for imbalance!** SMOTE and `class_weight='balanced'` both address the same problem — if you use SMOTE, set `class_weight=None` in your models. Stacking both effectively over-penalizes the majority class, distorting predictions. Our `train.py` handles this via the `--no-smote` flag:
- With SMOTE → models use `class_weight=None` (resampled data already handles imbalance)
- Without SMOTE (`--no-smote`) → models use `class_weight='balanced'` (loss function handles imbalance)

**Alternatives to discuss in interview:**
- `class_weight='balanced'` in the model (simpler, no synthetic data, no risk of overfitting to synthetic samples)
- Random undersampling of majority class (loses information)
- ADASYN (adaptive synthetic sampling)
- Ensemble methods like BalancedRandomForest

## 3. Train Models & Cross-Validate

In [ ]:
models = get_models()
cv_results = cross_validate_models(X_train_np, y_train_np, models, cv_folds=5)

In [ ]:
# Visualize cross-validation results
metrics = ['auprc', 'auroc', 'f1', 'precision', 'recall']
fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5))

for ax, metric in zip(axes, metrics):
    key = f'test_{metric}'
    data = {name: cv_results[name][key] for name in cv_results}
    ax.boxplot(data.values(), labels=data.keys())
    ax.set_title(metric.upper())
    ax.set_ylim([0, 1.05])
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Cross-Validation Results Across Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Train all models on full training set for comparison
trained_models = {}
for name in models:
    trained_models[name] = train_final_model(name, models, X_train_np, y_train_np)

## 4. Precision vs Recall Tradeoff

**The core question in fraud detection:**
- **High precision** = When we flag fraud, we're usually right (fewer false alarms) → better customer experience
- **High recall** = We catch most actual fraud (fewer missed fraud) → less financial loss

You can't have both. The decision threshold controls where you sit on this tradeoff.

**In an interview, explain:** The business decides the tradeoff. A bank losing $1M to fraud
may accept more false alarms. An e-commerce site with thin margins may prioritize precision.

In [ ]:
import os
output_dir = '../models'
os.makedirs(output_dir, exist_ok=True)

# Compare precision-recall tradeoff for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, trained_models.items()):
    y_scores = model.predict_proba(X_test_np)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_test_np, y_scores)
    
    ax.plot(thresholds, precisions[:-1], 'b-', label='Precision', linewidth=2)
    ax.plot(thresholds, recalls[:-1], 'r-', label='Recall', linewidth=2)
    
    # Mark best F1
    f1s = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1s)
    ax.axvline(x=thresholds[best_idx], color='green', linestyle='--', alpha=0.7,
               label=f'Best F1 @ {thresholds[best_idx]:.3f}')
    
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])

plt.suptitle('Precision vs Recall Tradeoff by Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Threshold Tuning

**The default 0.5 threshold is almost never optimal for imbalanced data.**

We find optimal thresholds for three strategies:
- **Best F1**: Balanced tradeoff between precision and recall
- **Best F2**: Weights recall 2x more (catch more fraud, accept more false alarms)
- **Recall ≥ 95%**: Ensure we catch at least 95% of fraud

In [ ]:
# Use the best model (XGBoost) for detailed threshold analysis
best_model = trained_models['XGBoost']
y_scores = best_model.predict_proba(X_test_np)[:, 1]

# Compare default threshold vs tuned
print('=== Default Threshold (0.5) ===')
y_pred_default = (y_scores >= 0.5).astype(int)
print(classification_report(y_test_np, y_pred_default, target_names=['Legit', 'Fraud'], digits=4))

threshold_results = tune_threshold(y_test_np, y_scores, output_dir)

In [ ]:
# Visual comparison of thresholds
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

thresholds_to_compare = {
    'Default (0.5)': 0.5,
    'Best F1': threshold_results['best_f1']['threshold'],
    'Best F2': threshold_results['best_f2']['threshold'],
    'Recall ≥ 95%': threshold_results['recall_95']['threshold'],
}

for ax, (label, t) in zip(axes, thresholds_to_compare.items()):
    y_pred = (y_scores >= t).astype(int)
    cm = confusion_matrix(y_test_np, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{label}\nthreshold={t:.3f}\nP={tp/(tp+fp):.3f} R={tp/(tp+fn):.3f}', fontsize=10)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices at Different Thresholds (XGBoost)', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 6. ROC Curve vs PR Curve

**Why PR curves are better for imbalanced data:**

- **ROC curve** plots TPR vs FPR. With 99.8% negatives, even 1000 false positives barely move FPR → ROC looks great even for mediocre models.
- **PR curve** plots Precision vs Recall. Precision directly feels the impact of false positives because the denominator (TP+FP) is small.
- **Random baseline**: ROC baseline is always 0.5 (diagonal). PR baseline equals the prevalence (~0.17%), making it easier to see actual model lift.

**Interview tip:** "AUROC can be misleadingly high on imbalanced data. AUPRC is a more honest metric."

In [ ]:
# Compare ROC and PR curves for all models
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
baseline = y_test_np.mean()

colors = ['steelblue', 'forestgreen', 'crimson']

for (name, model), color in zip(trained_models.items(), colors):
    y_scores_m = model.predict_proba(X_test_np)[:, 1]
    
    # ROC
    fpr, tpr, _ = roc_curve(y_test_np, y_scores_m)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={roc_auc:.4f})')
    
    # PR
    prec, rec, _ = precision_recall_curve(y_test_np, y_scores_m)
    pr_auc = average_precision_score(y_test_np, y_scores_m)
    ax2.plot(rec, prec, color=color, linewidth=2, label=f'{name} (AUPRC={pr_auc:.4f})')

# ROC formatting
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curves — All Models', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# PR formatting
ax2.axhline(y=baseline, color='k', linestyle='--', alpha=0.5, label=f'Random ({baseline:.4f})')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curves — All Models', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n=== Key Observation ===')
print('Notice how all ROC AUCs are above 0.97 — they all look great!')
print('But the PR curves tell a different story: the models vary more in AUPRC.')
print('This is why AUPRC is the preferred metric for imbalanced classification.')

## 7. Final Evaluation with Optimal Threshold

In [ ]:
best_threshold = threshold_results['best_f1']['threshold']
eval_results = full_evaluation(y_test_np, y_scores, best_threshold, output_dir)

print(f"\n=== Summary ===")
print(f"Model: XGBoost")
print(f"Threshold: {best_threshold:.4f}")
print(f"Precision: {eval_results['precision']:.4f}")
print(f"Recall: {eval_results['recall']:.4f}")
print(f"F1 Score: {eval_results['f1']:.4f}")

## 8. Interview Talking Points

### Why not just use accuracy?
With 99.83% legitimate transactions, a model predicting all-legitimate gets 99.83% accuracy but catches zero fraud.

### Precision vs Recall — what would you optimize for?
It depends on the business context. For a bank, missed fraud (low recall) means direct financial loss. For an e-commerce site, too many false positives (low precision) means blocked customers and lost revenue. Use the F-beta score with beta > 1 to weight recall higher.

### Why is 0.5 a bad default threshold?
The model outputs calibrated probabilities based on the training distribution. With SMOTE, the training distribution is artificial. Even without SMOTE, the optimal threshold for the business metric rarely coincides with 0.5.

### When is ROC misleading?
ROC uses FPR = FP/(FP+TN). With ~284K true negatives, adding 100 false positives barely moves FPR. So ROC says the model is great at separating classes, but in practice those 100 false positives matter a lot. PR curve doesn't have this blind spot because precision = TP/(TP+FP) is directly affected.

### SMOTE vs class_weight?
SMOTE creates synthetic minority samples, which can help models learn the decision boundary better. But it also risks overfitting to synthetic examples. `class_weight='balanced'` is simpler — it adjusts the loss function to penalize minority-class errors more heavily. In practice, try both and compare with cross-validation.

**Critical:** these two techniques are **not complementary** — never use both at the same time. SMOTE rebalances the data distribution; `class_weight='balanced'` reweights the loss function based on class frequencies. If the data is already SMOTE'd, the model sees a near-balanced distribution and applies balanced weights on top, double-correcting and over-weighting the minority class. Pick one strategy and set the other to neutral.